# Deriving Systems: the GPU labs

Chapters 7 and 8 are about a ratio between a GPU's compute throughput and its
memory bandwidth. You need a GPU to check them, and Colab's free T4 is enough.

**Before running anything:** Runtime, then Change runtime type, then pick a GPU.
The first cell will tell you if you forgot.

These labs do not trust the datasheet. They measure your card's bandwidth and
its matmul throughput and divide the two to get *your* ridge point. The claims
they check are ratios and shapes, so they hold on a T4, an A100, or a laptop
card, even though the absolute numbers differ by an order of magnitude.

In [ ]:
# 1. Confirm a GPU is attached, and see which one you drew.
import torch
if not torch.cuda.is_available():
    raise SystemExit('No GPU. Runtime > Change runtime type > T4 GPU, then rerun.')
p = torch.cuda.get_device_properties(0)
print(f'{p.name}, {p.total_memory/1e9:.1f} GB, compute capability {p.major}.{p.minor}')
print(f'torch {torch.__version__}, cuda {torch.version.cuda}')

In [ ]:
# 2. Get the labs. Pinning to a commit makes your run attributable to
#    a version of the code; drop the checkout line to take latest.
!git clone --quiet https://github.com/Venugopalan2610/perfbook.git 2>/dev/null || echo 'already cloned'
%cd /content/perfbook/experiments-gpu
!git log --oneline -1

In [ ]:
# 3. Chapter 7. Measures this card's ridge point, then walks a batch
#    size toward it. Watch the '% of peak' column.
!PERFBOOK_RESULTS=/content/results.json python 07_roofline.py

In [ ]:
# 4. Chapter 8. Allocates a real KV cache and asks the allocator what
#    it cost, instead of trusting the multiplication.
!PERFBOOK_RESULTS=/content/results.json python 08_kv_cache.py

## Predict before you scroll

The book's whole method is committing to a number before measuring, so before
you read the output above, write down:

1. Your card's ridge point, in FLOP per byte. The book quotes ~156 for an A100.
2. The arithmetic intensity at batch 1. The book says about 1.
3. What percentage of the card's peak you will actually get at batch 1.

Being wrong by 10x on the third one is the normal outcome, and it is the
reason chapter 7 exists.

## results.json

Both labs append their environment and every claim's outcome to
`/content/results.json`. Download it if you want to compare cards with someone:
it carries the GPU name, the measured constants, and which claims held.